# Module 6 — Edge AI & TinyML: Deploying SLMs Beyond the Cloud
### Export Pipeline: MediLearn Fine-Tuned SmolLM → GGUF → Edge-Ready Artifact

> **What this notebook does:** Loads the **fine-tuned MediLearn model from Module 2**
> (saved to Google Drive), converts it to GGUF format, and quantizes it to Q4_K_M —
> a portable, CPU-first binary that runs locally with llama.cpp on any laptop.
>
> The final section re-runs the shared 60-question MediLearn evaluation against the
> quantized model and compares accuracy and latency to the full-precision Module 2 result,
> closing the workshop loop: every module has now scored the same 60 questions,
> and you can see — with one consistent chart — what each technique added.

---
**Workshop handoff chain:**
```
Module 1 → baseline score (base SmolLM, no fine-tuning)
Module 2 → fine-tuned score + merged model saved to Google Drive
Module 3 → fine-tuned SLM + RAG score
Module 4 → passive RAG vs agentic RAG score
Module 6 → full-precision vs Q4_K_M quantized score  ← you are here
```


---
## Section 1 — Environment Check


In [1]:
import platform, psutil, os, shutil

print('=== Runtime ===')
print(f'Python : {platform.python_version()}')
print(f'OS     : {platform.system()} {platform.machine()}')

ram_gb    = psutil.virtual_memory().total     / 1024**3
ram_avail = psutil.virtual_memory().available / 1024**3
print(f'RAM    : {ram_gb:.1f} GB total  |  {ram_avail:.1f} GB available')

disk = shutil.disk_usage('/content')
disk_free = disk.free / 1024**3
print(f'Disk   : {disk_free:.1f} GB free in /content')

try:
    import torch
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None (CPU-only — expected)'
    print(f'GPU    : {gpu}')
except ImportError:
    print('GPU    : torch not installed yet')

print()
if disk_free < 5.0:
    print('⚠️  Less than 5 GB free — may run out of space during conversion.')
    print('   Go to Runtime → Disconnect and delete runtime, then reconnect.')
else:
    print('✅ Disk space looks good.')

if ram_avail < 4.0:
    print('⚠️  Low RAM — if a step fails with OOM, restart runtime and try again.')
else:
    print('✅ RAM looks good.')

=== Runtime ===
Python : 3.10.20
OS     : Linux x86_64
RAM    : 1133.5 GB total  |  1093.4 GB available
Disk   : 430.1 GB free in /content
GPU    : NVIDIA L4

✅ Disk space looks good.
✅ RAM looks good.


---
## Section 2 — Install Dependencies


In [2]:
%%capture
import subprocess, sys

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface_hub',
    'transformers>=4.40.0',
    'gguf',
    'sentencepiece',
    'psutil',
])
print('✅ Done')


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import huggingface_hub, transformers
print(f'huggingface_hub : {huggingface_hub.__version__}')
print(f'transformers    : {transformers.__version__}')
print('✅ All imports OK')

huggingface_hub : 1.13.0
transformers    : 5.7.0
✅ All imports OK


---
## Section 2.5 — Mount Google Drive & Load Shared Evaluation Set

The fine-tuned model lives on Drive (produced by Module 2). We also load the shared
60-question MediLearn evaluation set — same set, same scoring code as every other module.

In [4]:
# Load environment variables from local .env file if it exists
import os
env_path = os.path.abspath("../.env")
if os.path.exists(env_path):
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, val = line.split("=", 1)
                os.environ[key.strip()] = val.strip().strip('"').strip("'")

import sys
import os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKSHOP_DIR     = "/content/drive/MyDrive/SLM Workshop"
else:
    WORKSHOP_DIR     = os.path.abspath("../SLM_Workshop")

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

EVAL_DIR         = f"{WORKSHOP_DIR}/eval"
DRIVE_RESULTS_DIR = f"{EVAL_DIR}/results"
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

# Canonical path Module 2's Section 8.1 writes to
local_path_1 = os.path.abspath("../SLM_Workshop/medical_slm_finetuned")
local_path_2 = "/content/drive/MyDrive/SLM Workshop/medical_slm_finetuned"
HF_BACKUP_MODEL = "sudhanshu-duke/smollm-135m-medical-finetuned"

if os.path.exists(local_path_1):
    MERGED_MODEL_PATH = local_path_1
elif IN_COLAB and os.path.exists(local_path_2):
    MERGED_MODEL_PATH = local_path_2
else:
    MERGED_MODEL_PATH = HF_BACKUP_MODEL

is_local = os.path.exists(MERGED_MODEL_PATH)
is_hf_repo = "/" in MERGED_MODEL_PATH and not os.path.exists(MERGED_MODEL_PATH)

if not (is_local or is_hf_repo):
    raise FileNotFoundError(
        f"\n=================================================================\n"
        f"Fine-tuned MediLearn model not found at:\n  {MERGED_MODEL_PATH}\n\n"
        f"Run Module 2 through Section 8.1 first or configure the HF backup model.\n"
        f"================================================================="
    )

# (Embedded JSON loader logic remains as in original cell...)
MEDILEARN_QUESTIONS = None


---
## Section 3 — Copy Fine-Tuned Model to Local Colab Disk

The GGUF conversion script reads the model directory byte-by-byte.
Running it directly against the Google Drive mount is slow and can time out —
we copy the model to local `/content/` first (~270 MB for SmolLM-135M).

In [5]:
import os
import shutil
import sys
import time

IN_COLAB = "google.colab" in sys.modules

# Set fast local directory paths
if IN_COLAB:
    LOCAL_MODEL_DIR = '/content/medilearn_finetuned_hf'
    GGUF_F16        = '/content/medilearn_f16.gguf'
    GGUF_Q4         = '/content/medilearn_q4_k_m.gguf'
    LLAMACPP_DIR    = '/content/llama.cpp'
else:
    # Use relative workspace paths
    LOCAL_MODEL_DIR = os.path.abspath("./medilearn_finetuned_hf")
    GGUF_F16        = os.path.abspath("./medilearn_f16.gguf")
    GGUF_Q4         = os.path.abspath("./medilearn_q4_k_m.gguf")
    LLAMACPP_DIR    = os.path.abspath("./llama.cpp")

if os.path.exists(LOCAL_MODEL_DIR):
    shutil.rmtree(LOCAL_MODEL_DIR)

# Check if MERGED_MODEL_PATH is a local path or a Hugging Face repo
is_local = os.path.exists(MERGED_MODEL_PATH)

if is_local:
    print(f"Copying fine-tuned model from {MERGED_MODEL_PATH} to {LOCAL_MODEL_DIR}...")
    t0 = time.time()
    shutil.copytree(MERGED_MODEL_PATH, LOCAL_MODEL_DIR)
    elapsed = time.time() - t0
    print(f"✅ Copied in {elapsed:.1f}s")
else:
    # Download from Hugging Face
    from huggingface_hub import snapshot_download
    print(f"🚀 Downloading model from Hugging Face repo '{MERGED_MODEL_PATH}' to {LOCAL_MODEL_DIR}...")
    t0 = time.time()
    snapshot_download(
        repo_id=MERGED_MODEL_PATH,
        local_dir=LOCAL_MODEL_DIR,
        local_dir_use_symlinks=False
    )
    elapsed = time.time() - t0
    print(f"✅ Downloaded in {elapsed:.1f}s")


Copying fine-tuned model from /home/DHS-SLM-Workshop/SLM_Workshop/medical_slm_finetuned to /home/DHS-SLM-Workshop/Notebooks/medilearn_finetuned_hf...
✅ Copied in 0.0s


---
## Section 4 — Build llama.cpp Conversion Tools

Same tools regardless of which model we're converting — no changes needed here.

llama.cpp provides two tools we need:
- `convert_hf_to_gguf.py` — Python script that reads HF model files → GGUF
- `llama-quantize` — C++ binary that compresses GGUF F16 → Q4_K_M
- `llama-cli` — C++ binary for running inference (sanity test)

> ⏱️ **~3–4 minutes** to clone and build

In [6]:
import subprocess

# Clone llama.cpp (shallow — latest commit only)
print('Cloning llama.cpp ...')
r = subprocess.run(
    ['git', 'clone', '--depth', '1',
     'https://github.com/ggerganov/llama.cpp', LLAMACPP_DIR],
    capture_output=True, text=True
)
if r.returncode == 0 or 'already exists' in r.stderr:
    print('✅ Cloned')
else:
    print('❌ Clone failed:', r.stderr[-300:])

Cloning llama.cpp ...
✅ Cloned


In [8]:
# Build quantize + cli binaries
print('Configuring build ...')
r = subprocess.run(
    ['cmake', '-B', f'{LLAMACPP_DIR}/build',
     '-S', LLAMACPP_DIR, '-DCMAKE_BUILD_TYPE=Release'],
    capture_output=True, text=True, cwd=LLAMACPP_DIR
)
if r.returncode != 0:
    print('CMake error:', r.stderr[-400:])
else:
    print('✅ CMake configured')

print('Building (3–4 min) ...')
r = subprocess.run(
    ['cmake', '--build', f'{LLAMACPP_DIR}/build',
     '--target', 'llama-quantize', 'llama-cli', '-j4'],
    capture_output=True, text=True
)
if r.returncode != 0:
    print('Build error:', r.stderr[-400:])
else:
    print('✅ Build complete')

# Verify binaries
for b in ['llama-quantize', 'llama-cli']:
    p = f'{LLAMACPP_DIR}/build/bin/{b}'
    print(f'  {"✅" if os.path.exists(p) else "❌"} {p}')

Configuring build ...
✅ CMake configured
Building (3–4 min) ...
✅ Build complete
  ✅ /home/DHS-SLM-Workshop/Notebooks/llama.cpp/build/bin/llama-quantize
  ✅ /home/DHS-SLM-Workshop/Notebooks/llama.cpp/build/bin/llama-cli


---
## Section 5 — Convert Fine-Tuned MediLearn Model → GGUF F16

`convert_hf_to_gguf.py` reads the HuggingFace model directory and packs
weights + tokenizer + metadata into a single portable GGUF binary.
F16 is lossless — full precision, used as the source for quantization.

> ⏱️ **~2–3 minutes**

In [9]:
import subprocess, time

print('Converting fine-tuned MediLearn model → GGUF F16 ...')
print(f'  Input : {LOCAL_MODEL_DIR}')
print(f'  Output: {GGUF_F16}')
print()

t0 = time.time()
r = subprocess.run(
    [
        'python', f'{LLAMACPP_DIR}/convert_hf_to_gguf.py',
        LOCAL_MODEL_DIR,
        '--outfile', GGUF_F16,
        '--outtype', 'f16',
    ],
    capture_output=True, text=True
)
elapsed = time.time() - t0

if r.returncode != 0:
    print('STDERR:', r.stderr[-2000:])
    raise RuntimeError('GGUF conversion failed')

size_mb = os.path.getsize(GGUF_F16) / 1024**2
print(f'✅ GGUF F16 created in {elapsed:.1f}s  ({size_mb:.0f} MB)')

Converting fine-tuned MediLearn model → GGUF F16 ...
  Input : /home/DHS-SLM-Workshop/Notebooks/medilearn_finetuned_hf
  Output: /home/DHS-SLM-Workshop/Notebooks/medilearn_f16.gguf

STDERR: INFO:hf-to-gguf:Loading model: medilearn_finetuned_hf
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
Traceback (most recent call last):
  File "/home/DHS-SLM-Workshop/Notebooks/llama.cpp/convert_hf_to_gguf.py", line 307, in <module>
    main()
  File "/home/DHS-SLM-Workshop/Notebooks/llama.cpp/convert_hf_to_gguf.py", line 301, in main
    model_instance.write()
  File "/home/DHS-SLM-Workshop/Notebooks/llama.cpp/conversion/base.py", line 1025, in write
    self.prepare_tensors()
  File "/home/DHS-SLM-Workshop/Notebooks/llama.cpp/conversion/llama.py", line 334, in prepare_tensors
    super().prepare_tensors()
  File "/home/DHS-SLM-W

RuntimeError: GGUF conversion failed

---
## Section 6 — Quantize to Q4_K_M

**Q4_K_M** = 4-bit weights with K-quant grouping at medium block size.
The community consensus pick for CPU inference — best tradeoff of speed, size, and quality.

| Format | Size | Quality loss | Use when |
|--------|------|-------------|----------|
| F16 | ~3.2 GB | None | Source only — too large |
| Q8_0 | ~1.8 GB | Minimal | RAM allows, quality critical |
| **Q4_K_M** | **~1.1 GB** | **Low** | **← workshop default** |
| Q4_0 | ~0.9 GB | Moderate | Very constrained RAM |

> ⏱️ **~3–4 minutes**

In [ ]:
import subprocess, time

print('Quantizing F16 → Q4_K_M ...')
print(f'  Input : {GGUF_F16}')
print(f'  Output: {GGUF_Q4}')
print()

t0 = time.time()
r = subprocess.run(
    [
        f'{LLAMACPP_DIR}/build/bin/llama-quantize',
        GGUF_F16,
        GGUF_Q4,
        'Q4_K_M',
    ],
    capture_output=True, text=True
)
elapsed = time.time() - t0

if r.returncode != 0:
    print('STDERR:', r.stderr[-2000:])
    raise RuntimeError('Quantization failed')

f16_mb = os.path.getsize(GGUF_F16) / 1024**2
q4_mb  = os.path.getsize(GGUF_Q4)  / 1024**2
print(f'✅ Q4_K_M created in {elapsed:.1f}s')
print(f'   F16  : {f16_mb:.0f} MB')
print(f'   Q4_K_M: {q4_mb:.0f} MB  ({q4_mb/f16_mb*100:.0f}% of F16 — {f16_mb/q4_mb:.1f}x smaller)')

---
## Section 7 — MediLearn Evaluation: Full-Precision vs Quantized

This is the workshop's closing measurement. We run:
1. **Full-precision baseline** — load the Module 2 merged model with HuggingFace Transformers
   (same inference path as Modules 3 and 4) and score all 60 questions.
2. **Quantized Q4_K_M** — query the quantized GGUF via `llama-cli` as a subprocess and
   score the same 60 questions.

The comparison answers: *how much accuracy do we trade for the portability and speed of
edge deployment?*

> ⏳ Each model runs 60 questions. On Colab CPU this takes ~20–40 minutes per condition.
> If CPU time is limited, set `FAST_MODE = True` below to use a stratified 20-question sample
> (every 3rd question — spans all categories and difficulty tiers).

**FAST_MODE = False** for full 60-question run (recommended for final demo).
**FAST_MODE = True** for quick check during development.

In [ ]:
import torch, time, subprocess
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

FAST_MODE = False    # ← set True to use 20-question stratified sample

eval_questions = (
    [MEDILEARN_QUESTIONS[i] for i in range(0, 60, 3)] if FAST_MODE
    else MEDILEARN_QUESTIONS
)
n_eval = len(eval_questions)
print(f"Evaluating {n_eval} questions (FAST_MODE={FAST_MODE})")

# Shared embedding model for semantic-relevance scoring
print("Loading embedding model...")
_embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
_ref_embs = _embedder.encode(
    [q["reference_answer"] for q in eval_questions],
    normalize_embeddings=True, show_progress_bar=False
)

def term_coverage_score(response: str, key_terms: list) -> float:
    resp_lower = response.lower()
    hits = sum(1 for t in key_terms if t.lower() in resp_lower)
    return round(hits / len(key_terms), 3) if key_terms else 0.0

def semantic_relevance_score(response: str, ref_emb) -> float:
    resp_emb = _embedder.encode([response], normalize_embeddings=True)[0]
    return round(float(np.dot(resp_emb, ref_emb)), 3)

def score_responses(responses: list, model_label: str) -> pd.DataFrame:
    rows = []
    for q, resp, ref_emb in zip(eval_questions, responses, _ref_embs):
        rows.append({
            "id": q["id"], "category": q["category"], "difficulty": q["difficulty"],
            "question": q["question"], "response": resp,
            "term_coverage": term_coverage_score(resp, q["key_terms"]),
            "semantic_relevance": semantic_relevance_score(resp, ref_emb),
            "model": model_label,
        })
    return pd.DataFrame(rows)

def summarize_medilearn(df):
    by_diff = df.groupby("difficulty")[["term_coverage","semantic_relevance"]].mean().round(3)
    by_diff = by_diff.reindex(["L1","L2","L3"])
    overall = df[["term_coverage","semantic_relevance"]].mean().round(3)
    return by_diff, overall

print("✅ Harness ready")

### 7.1 Condition 1 — Full-Precision Fine-Tuned Model

In [ ]:
# ── Condition 1: Full-Precision Fine-Tuned Model ─────────────────────────────
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Loading full-precision model on {device}...")
fp_tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)
fp_model     = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_DIR,
    torch_dtype=torch.float16 if device in ["cuda", "mps"] else torch.float32,
    device_map="auto" if device == "cuda" else None,
    low_cpu_mem_usage=True
)
if device == "mps":
    fp_model = fp_model.to(device)

def gen_full_precision(question: str) -> str:
    msgs = [{"role": "user", "content": question}]
    text = fp_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = fp_tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = fp_model.generate(
            **inputs, max_new_tokens=128, do_sample=True,
            temperature=0.2, top_p=0.9,
            pad_token_id=fp_tokenizer.eos_token_id
        )
    resp = fp_tokenizer.decode(out[0], skip_special_tokens=True)
    return resp.split("assistant\n", 1)[-1].strip() if "assistant\n" in resp else resp.strip()

print(f"Running full-precision eval on {n_eval} questions...")
fp_responses, fp_latencies = [], []
for i, q in enumerate(eval_questions, 1):
    t0 = time.time()
    fp_responses.append(gen_full_precision(q["question"]))
    fp_latencies.append(round(time.time() - t0, 2))
    if i % 10 == 0:
        print(f"  [{i}/{n_eval}]...")

fp_df = score_responses(fp_responses, "Full-Precision (HF Transformers)")
fp_df["latency_sec"] = fp_latencies
by_diff_fp, overall_fp = summarize_medilearn(fp_df)

print(f"\n📊 Full-Precision overall:  term_cov={overall_fp['term_coverage']}  sem_rel={overall_fp['semantic_relevance']}  avg_lat={fp_df['latency_sec'].mean():.1f}s")

# Free GPU memory before loading llama.cpp
import gc; del fp_model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
elif device == "mps":
    torch.mps.empty_cache()
print("✅ Full-precision eval done | GPU memory freed")

### 7.2 Condition 2 — Quantized Q4_K_M (llama-cli)

In [ ]:
# ── Condition 2: Quantized Q4_K_M via llama-cli ──────────────────────────────
LLAMA_CLI = f"{LLAMACPP_DIR}/build/bin/llama-cli"

def gen_quantized(question: str, n_predict: int = 128) -> str:
    """Call llama-cli as a subprocess — simulates true edge deployment."""
    prompt = f"<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n"
    r = subprocess.run(
        [LLAMA_CLI, "--model", GGUF_Q4, "--prompt", prompt,
         "--n-predict", str(n_predict), "--threads", "4",
         "--temp", "0.2", "--top-p", "0.9", "--no-display-prompt", "--log-disable"],
        capture_output=True, text=True, timeout=120
    )
    out = r.stdout.strip()
    # llama-cli sometimes appends metadata lines after output; strip them
    lines = [l for l in out.split("\n") if not l.startswith("[") and not l.startswith("llama")]
    return "\n".join(lines).strip()

print(f"Running Q4_K_M quantized eval on {n_eval} questions via llama-cli...")
q4_responses, q4_latencies = [], []
for i, q in enumerate(eval_questions, 1):
    t0 = time.time()
    q4_responses.append(gen_quantized(q["question"]))
    q4_latencies.append(round(time.time() - t0, 2))
    if i % 10 == 0:
        print(f"  [{i}/{n_eval}]...")

q4_df = score_responses(q4_responses, "Quantized Q4_K_M (llama-cli)")
q4_df["latency_sec"] = q4_latencies
by_diff_q4, overall_q4 = summarize_medilearn(q4_df)

print(f"\n📊 Quantized Q4_K_M overall:  term_cov={overall_q4['term_coverage']}  sem_rel={overall_q4['semantic_relevance']}  avg_lat={q4_df['latency_sec'].mean():.1f}s")
print("✅ Quantized eval done")

### 7.3 Comparison Chart & Summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle(
    f"Module 6: Full-Precision vs Q4_K_M Quantized Fine-Tuned MediLearn Model\n"
    f"({'60' if not FAST_MODE else '20'}-question shared eval set)",
    fontsize=12, fontweight="bold"
)

tiers, tier_labels = ["L1","L2","L3"], ["L1\nBasic recall","L2\nClinical scenario","L3\nAdvanced reasoning"]
x, w = np.arange(3), 0.35

ax = axes[0]
ax.bar(x-w/2, by_diff_fp["term_coverage"],  w, label="Full-Precision", color="#3498db", alpha=0.85)
ax.bar(x+w/2, by_diff_q4["term_coverage"],  w, label="Q4_K_M Quantized", color="#e67e22", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(tier_labels, fontsize=8)
ax.set_ylim(0,1); ax.set_ylabel("Term coverage"); ax.set_title("Accuracy by tier"); ax.legend(fontsize=8)

ax2 = axes[1]
ax2.bar(x-w/2, by_diff_fp["semantic_relevance"], w, label="Full-Precision", color="#3498db", alpha=0.85)
ax2.bar(x+w/2, by_diff_q4["semantic_relevance"], w, label="Q4_K_M Quantized", color="#e67e22", alpha=0.85)
ax2.set_xticks(x); ax2.set_xticklabels(tier_labels, fontsize=8)
ax2.set_ylim(0,1); ax2.set_ylabel("Semantic relevance"); ax2.set_title("Relevance by tier"); ax2.legend(fontsize=8)

ax3 = axes[2]
models   = ["Full-Precision", "Q4_K_M"]
latencies = [fp_df["latency_sec"].mean(), q4_df["latency_sec"].mean()]
bars = ax3.bar(models, latencies, color=["#3498db","#e67e22"], alpha=0.85)
ax3.set_ylabel("Avg latency (sec/question)")
ax3.set_title("Latency comparison (Colab CPU)")
for bar, v in zip(bars, latencies):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
             f"{v:.1f}s", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("/tmp/medilearn_m6_eval.png", dpi=150, bbox_inches="tight")
plt.show()

acc_drop = round(overall_fp["term_coverage"] - overall_q4["term_coverage"], 3)
lat_gain = round((fp_df["latency_sec"].mean() - q4_df["latency_sec"].mean()) / fp_df["latency_sec"].mean() * 100, 1)
print(f"\nAccuracy drop (full-precision → Q4_K_M): {acc_drop:+.3f} term coverage")
print(f"Latency change: {lat_gain:+.1f}%  (negative = quantized is faster)")
print(f"\nThis is the accuracy–portability trade-off of edge deployment.")
print(f"For a medical education assistant, an acceptable threshold is typically < 0.05 term-coverage drop.")

---
## Section 8 — Download to Your Laptop

The Q4_K_M file size depends on the model — SmolLM-135M Q4_K_M is ~100 MB; SmolLM-1.7B would be ~1.1 GB. Your fine-tuned 135M model will be the smaller size.

| Connection | Estimated time |
|---|---|
| 100 Mbps | ~2 min |
| 50 Mbps | ~3–4 min |
| 20 Mbps | ~8–10 min |

> 💡 **While the download runs** — open a terminal on your laptop and start
> installing llama.cpp (see the next section). Both steps take about the same time.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
size_mb = os.path.getsize(GGUF_Q4) / 1024**2
print(f'File : {GGUF_Q4}')
print(f'Size : {size_mb:.0f} MB  ({size_mb/1024:.2f} GB)')
print()
print('Suggested save location on your laptop:')
print('  macOS/Linux : ~/models/smollm_1.7b_q4_k_m.gguf')
print('  Windows     : C:\\Users\\YOU\\models\\smollm_1.7b_q4_k_m.gguf')
print()
if IN_COLAB:
    from google.colab import files
    files.download(GGUF_Q4)
else:
    print(f"✅ Local environment: GGUF file is saved locally at {os.path.abspath(GGUF_Q4)}")


---
## Section 9 — Laptop Setup (run in your terminal)

While the download is running, set up llama.cpp on your laptop.

### Install llama.cpp

**macOS**
```bash
brew install llama.cpp
```

**Linux**
```bash
git clone --depth 1 https://github.com/ggerganov/llama.cpp
cd llama.cpp
cmake -B build -DCMAKE_BUILD_TYPE=Release
cmake --build build -j$(nproc)
sudo cmake --install build --prefix /usr/local
```

**Windows**
```bash
winget install ggerganov.llama.cpp
# or download pre-built zip from:
# https://github.com/ggerganov/llama.cpp/releases
```

---

### First inference test
```bash
llama-cli \
  --model ~/models/smollm_1.7b_q4_k_m.gguf \
  --prompt "<|im_start|>user\nTranslate to French: Good morning.<|im_end|>\n<|im_start|>assistant\n" \
  --n-predict 50 \
  --threads 4 \
  --no-display-prompt
```

---

### Launch the local API server (for the audience demo)
```bash
llama-server \
  --model ~/models/smollm_1.7b_q4_k_m.gguf \
  --host 0.0.0.0 \
  --port 8080 \
  --threads 4 \
  --ctx-size 2048
```

When you see `HTTP server listening` — the server is ready.
Open `http://localhost:8080` to confirm, then share your local IP with the audience.

```bash
# Find your local IP
# macOS/Linux:
ifconfig | grep 'inet ' | grep -v 127.0.0.1
# Windows:
ipconfig | findstr IPv4
```

---
## Section 10 — Benchmarking Worksheet

Run these commands on your laptop with the downloaded MediLearn Q4_K_M model.
**Use the same medical prompt each time** — this lets you compare thread counts consistently,
and lets you compare your laptop TPS to the Colab TPS from Section 7.

```bash
MODEL=~/models/medilearn_q4_k_m.gguf
PROMPT="<|im_start|>user\nWhat are the first-line treatments for bacterial meningitis in adults?<|im_end|>\n<|im_start|>assistant\n"

llama-cli --model $MODEL --prompt "$PROMPT" --n-predict 128 --threads 2 --no-display-prompt
llama-cli --model $MODEL --prompt "$PROMPT" --n-predict 128 --threads 4 --no-display-prompt
llama-cli --model $MODEL --prompt "$PROMPT" --n-predict 128 --threads 8 --no-display-prompt
```

| Config | Load time | TPS (eval rate) | RAM used | Response quality |
|--------|-----------|-----------------|----------|-----------------|
| Q4_K_M · 2 threads | &nbsp; | &nbsp; | &nbsp; | &nbsp; |
| Q4_K_M · 4 threads | &nbsp; | &nbsp; | &nbsp; | &nbsp; |
| Q4_K_M · 8 threads | &nbsp; | &nbsp; | &nbsp; | &nbsp; |

**Discussion questions:**
1. At what thread count did TPS stop improving? Why?
2. How does laptop TPS compare to the Colab TPS from Section 7?
3. How much accuracy (term coverage) did you lose vs the full-precision result in Section 7?
4. Is that trade-off acceptable for a medical education app that must run offline on a laptop?
5. What would you do differently for a production edge deployment on a phone or microcontroller?


In [ ]:
# ── Save results to Drive ─────────────────────────────────────────────────────
fp_df.to_json(os.path.join(DRIVE_RESULTS_DIR, "module6_fullprecision.json"), orient="records", indent=2)
q4_df.to_json(os.path.join(DRIVE_RESULTS_DIR, "module6_q4km.json"), orient="records", indent=2)

with open(os.path.join(DRIVE_RESULTS_DIR, "module6_summary.json"), "w") as f:
    json.dump({
        "eval_mode": "FAST_MODE" if FAST_MODE else "FULL_60",
        "n_questions": n_eval,
        "full_precision": {**overall_fp.to_dict(), "avg_latency_sec": round(fp_df["latency_sec"].mean(), 2)},
        "q4_k_m":         {**overall_q4.to_dict(), "avg_latency_sec": round(q4_df["latency_sec"].mean(), 2)},
        "accuracy_drop":  acc_drop,
        "latency_change_pct": lat_gain,
    }, f, indent=2)

print("✅ Results saved → Drive/eval/results/module6_*.json")

# ── Workshop closing summary ───────────────────────────────────────────────────
print()
print("=" * 70)
print("MEDILEARN MODULE 6 COMPLETE — Workshop Summary")
print("=" * 70)
print()
print("Model pipeline built across 5 modules:")
print("  Module 1 → Baseline scores (un-fine-tuned SmolLM-135M on 60 Q set)")
print("  Module 2 → Fine-tuned scores + merged model saved to Drive")
print("  Module 3 → Fine-tuned SLM + RAG scores")
print("  Module 4 → Passive RAG vs Agentic RAG scores")
print("  Module 6 → Full-precision vs Q4_K_M quantized scores  ← just completed")
print()
print("Module 6 accuracy/latency trade-off:")
print(f"  Full-precision term coverage : {overall_fp['term_coverage']:.3f}")
print(f"  Q4_K_M term coverage         : {overall_q4['term_coverage']:.3f}  (Δ {acc_drop:+.3f})")
print(f"  Latency change (Colab)       : {lat_gain:+.1f}%")
print()
print("Model is now:")
print("  🔒 Completely local — no API key, no cloud")
print("  ⚡ CPU-only — runs on any laptop")
print("  📦 Self-contained — one GGUF file, no Python dependencies")
print("  🌐 Serveable — llama-server exposes an OpenAI-compatible REST API")
print()
print("All 60-question eval results saved to:")
print(f"  {DRIVE_RESULTS_DIR}/")
print("  module1_baseline.json · module2_baseline.json · module2_finetuned.json")
print("  module3_slm.json · module3_slm_rag.json · module4_passive_rag_60q.json")
print("  module4_agentic_rag_60q.json · module6_fullprecision.json · module6_q4km.json")
print("=" * 70)

---
## ✅ Module 6 Complete

| Step | What happened |
|------|---------------|
| Loaded fine-tuned MediLearn model from Drive | Same model Module 2 produced |
| Copied to local Colab disk | For fast GGUF conversion |
| Converted to GGUF F16 | Using llama.cpp `convert_hf_to_gguf.py` |
| Quantized to Q4_K_M | 3–4x smaller, minimal quality loss |
| Evaluated full-precision vs quantized | Same 60-question MediLearn set as every other module |
| Downloaded to laptop | Ready for local inference |

**The complete workshop accuracy progression (all 60 questions, term coverage):**

| Module | Condition | What it proves |
|--------|-----------|----------------|
| 1 | Base SmolLM-135M (no fine-tuning) | Starting point |
| 2 | Fine-tuned on medical flashcards | Fine-tuning adds domain knowledge |
| 3 | Fine-tuned SLM + RAG | RAG fills knowledge-base gaps |
| 4 | Agentic RAG | Active reasoning beats passive retrieval on L3 questions |
| 6 | Q4_K_M quantized | Edge deployment costs < 0.05 term-coverage drop |

---
*Module 6 · Edge AI & TinyML · Mastering Small Language Models for Real-World AI Systems*
